# Cross-Market Transfer Learning for Stock Return Prediction — Version 3

Can a model pretrained on U.S. stocks help predict next-month returns in the United Kingdom and Japan when local training
data are limited? The data are monthly stock-level observations from the Jensen–Kelly–Pedersen (JKP) Global Factor dataset.

This notebook is the complete implementation: data, training, tuning, selection, forecasts and evaluation. Versions 1 and 2 are
preserved under the Git tags `v1.0` and `v2.0-experiment` and the branch `v2-finalization`.

**Status of the evidence**
- Tuning uses development folds that end in December 2017. Checkpoints and the final model use 2018–2019.
- The 2018–2019 results of Version 2 were known when the configurations below were chosen, and the test results of 2020–2025
  have been examined repeatedly. The test period is therefore **exploratory**. Generalization to new data is unverified.
- No stage runs by default, so **Run All trains nothing and reads no data**.

| Stage | Switch | Needs | Device |
|---|---|---|---|
| Data checks | `RUN_DATA_CHECKS` | Parquet files | CPU |
| Baselines | `RUN_BASELINES` | data checks | CPU |
| Screening | `RUN_SCREENING` | data checks | GPU |
| Finalist choice | `RUN_FINALIST_SELECTION` | screening of all configurations | CPU |
| Finalist training | `RUN_FINAL_TRAINING` | finalist choice | GPU |
| Model selection and test forecasts | `RUN_PREDICTIONS` | baselines, finalist training | GPU or CPU |
| Test evaluation | `RUN_FINAL_EVALUATION` | predictions | CPU |

To run a stage, set its switch to `True` in Section 1 and use Run All. On Amarel the job scripts set the same switches.

## 1. Imports, Paths and Configuration

In [ ]:
import hashlib  # checksums and fingerprints
import json  # run records
import os  # settings passed by job scripts
import sys  # operating system and Python version
import time  # training time
import warnings  # convergence warnings
from importlib.metadata import version  # installed package versions
from itertools import product  # parameter grids
from pathlib import Path  # portable file paths

import joblib  # save baseline models
import matplotlib.pyplot as plt  # figures
import numpy as np  # numerical arrays
import pandas as pd  # tables and dates
import pyarrow.parquet as pq  # Parquet schemas and metadata
from xgboost import XGBRegressor  # boosted trees; must be imported before torch on macOS
import torch  # neural networks and GPU support
import torch.nn.functional as F  # attention and losses
from IPython.display import display  # formatted tables
from sklearn.exceptions import ConvergenceWarning  # raised when Elastic Net does not converge
from sklearn.linear_model import ElasticNet  # linear baseline
from torch import nn  # neural network layers

In [ ]:
# Execution switches. Everything is off by default.
RUN_DATA_CHECKS = False         # integrity, data quality and the predictor list (run once, first)
RUN_BASELINES = False           # Elastic Net and XGBoost
RUN_SCREENING = False           # Stage A: every configuration on the development folds
RUN_FINALIST_SELECTION = False  # choose the finalists from complete screening results
RUN_FINAL_TRAINING = False      # Stage B: finalists on the full training data
RUN_PREDICTIONS = False         # validation leaderboard, locked selection, then test forecasts
RUN_FINAL_EVALUATION = False    # test metrics, tests and portfolios

ONLY_CONFIGS = []               # screening: empty = all; e.g. ["control"] to run one
ONLY_FINALIST = None            # final training: None = all; 0, 1 or 2 to run one
MAX_EPOCHS = 100                # at most 100; shortened runs get their own results folder
DEVICE_CHOICE = "auto"          # cuda, then mps, then cpu

In [ ]:
def switch(name, default):
    """Read an on/off environment variable ("1" or "0")."""
    return {"1": True, "0": False}[os.environ[name]] if name in os.environ else default


RUN_DATA_CHECKS = switch("CMP_DATA_CHECKS", RUN_DATA_CHECKS)
RUN_BASELINES = switch("CMP_BASELINES", RUN_BASELINES)
RUN_SCREENING = switch("CMP_SCREENING", RUN_SCREENING)
RUN_FINALIST_SELECTION = switch("CMP_FINALIST_SELECTION", RUN_FINALIST_SELECTION)
RUN_FINAL_TRAINING = switch("CMP_FINAL_TRAINING", RUN_FINAL_TRAINING)
RUN_PREDICTIONS = switch("CMP_PREDICTIONS", RUN_PREDICTIONS)
RUN_FINAL_EVALUATION = switch("CMP_FINAL_EVALUATION", RUN_FINAL_EVALUATION)
ONLY_CONFIGS = [name for name in os.environ.get("CMP_CONFIGS", ",".join(ONLY_CONFIGS)).split(",") if name]
if os.environ.get("CMP_FINALIST", "") != "":
    ONLY_FINALIST = int(os.environ["CMP_FINALIST"])
MAX_EPOCHS = int(os.environ.get("CMP_EPOCHS", MAX_EPOCHS))
DEVICE_CHOICE = os.environ.get("CMP_DEVICE", DEVICE_CHOICE)
if not 1 <= MAX_EPOCHS <= 100:
    raise ValueError("MAX_EPOCHS must be between 1 and 100.")

STAGES = {
    "data checks": RUN_DATA_CHECKS, "baselines": RUN_BASELINES, "screening": RUN_SCREENING,
    "finalist selection": RUN_FINALIST_SELECTION, "final training": RUN_FINAL_TRAINING,
    "predictions": RUN_PREDICTIONS, "final evaluation": RUN_FINAL_EVALUATION,
}
ANY_STAGE = any(STAGES.values())
NEEDS_FEATURES = RUN_BASELINES or RUN_SCREENING or RUN_FINAL_TRAINING or RUN_PREDICTIONS

PROJECT_ROOT = Path(os.environ.get("CMP_PROJECT_ROOT", "."))
DATA_DIR = Path(os.environ.get("CMP_DATA_DIR", PROJECT_ROOT / "data"))
RESULTS_DIR = PROJECT_ROOT / ("results_v3" if MAX_EPOCHS == 100 else f"results_v3_{MAX_EPOCHS}epochs")
RUNS_DIR, TABLES_DIR = RESULTS_DIR / "runs", RESULTS_DIR / "tables"

if sys.platform == "darwin":
    torch.set_num_threads(1)  # avoids a macOS crash between the PyTorch and XGBoost thread libraries
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # reproducible CUDA matrix products
torch.use_deterministic_algorithms(True, warn_only=True)     # some GPU operations may still differ slightly
pd.set_option("display.float_format", "{:,.4f}".format)

print("Stages: " + (", ".join(name for name, on in STAGES.items() if on) or "none (definitions only)"))

**Research constants.** Version 2's selection metrics differed by less than their uncertainty, so `EQUIVALENT_IC` treats candidates
within 0.001 as equal and then prefers the model with fewer parameters.

In [ ]:
CODE_VERSION = 1  # increase after any change that affects trained models, so older runs are rejected
SEED = 42
SCREENING_SEEDS, FINAL_SEEDS = [42, 43], [42, 43, 44]  # the first seed is also used to choose the learning rate

COUNTRIES, LOCAL_COUNTRIES = ["USA", "GBR", "JPN"], ["GBR", "JPN"]
WINDOWS = {"train_1y": 12, "train_3y": 36, "train_5y": 60}  # months of local training
TARGET = "ret_exc_lead1m"
SCREENS = ["common", "exch_main", "primary_sec", "obs_main"]
METADATA = ["id", "eom", "excntry", "gvkey", "permno", "size_grp", "me", TARGET] + SCREENS
FILES = {country: DATA_DIR / f"{country}.parquet" for country in COUNTRIES}
EXPECTED_ROWS = {"USA": 1_604_094, "GBR": 534_295, "JPN": 1_175_157}
AUDITED_FINGERPRINTS = {  # files verified by the Version 2 audit
    "USA": "4e28c30a85cb1cad55af8ac06b5f563e4bfc72a344a29e268c47c15f1f436672",
    "GBR": "e57f57695a2b03ac98faf007b1fd5fb985e0aa0310901d8a5c7ada655f3083be",
    "JPN": "3669f62de3069594e99b6c14e93ad8ca8995b2fec4b48602f3a17d9867c6980c",
}
KEYS = ["id", "eom"]

EQUIVALENT_IC = 0.001       # validation differences below this are treated as ties
MSE_TOLERANCE = 0.0025      # calibrated raw and clipped MSE may each be at most 0.25% worse than a zero forecast in a country
CLIP_QUANTILES = (0.005, 0.995)
BOOTSTRAP_SAMPLES, BLOCK_MONTHS = 10_000, 6
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

In [ ]:
def save_safely(path, save):
    """Write to a temporary file first, so an interrupted job never leaves a partial file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.name}.{os.getpid()}.tmp")
    save(temporary)
    temporary.replace(path)


def save_json(path, value):
    save_safely(path, lambda temporary: temporary.write_text(json.dumps(value, indent=1)))


def save_table(table, name):
    """Save an aggregate table (no stock-level data)."""
    save_safely(TABLES_DIR / f"{name}.csv", lambda temporary: table.to_csv(temporary, index=False))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while block := file.read(1 << 20):
            digest.update(block)
    return digest.hexdigest()


def short_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]


def dataset_fingerprint(path):
    """Hash of the Parquet footer and file size: quick, but not a full checksum."""
    size = path.stat().st_size
    with open(path, "rb") as file:
        file.seek(-8, os.SEEK_END)
        length = int.from_bytes(file.read(4), "little")
        file.seek(-8 - length, os.SEEK_END)
        return hashlib.sha256(file.read(length) + str(size).encode()).hexdigest()


def environment_details():
    """Where a result was produced. Git values are resolved on the submission host and inherited by the job."""
    cuda = torch.cuda.is_available()
    return {
        "git_commit": os.environ.get("CMP_GIT_COMMIT", "local run"), "git_dirty": os.environ.get("CMP_GIT_DIRTY", "unknown"),
        "notebook_sha256": os.environ.get("CMP_NOTEBOOK_SHA256", "unknown"), "slurm_job": os.environ.get("SLURM_JOB_ID", ""),
        "python": sys.version.split()[0], "gpu": torch.cuda.get_device_name() if cuda else "", "recorded_at": pd.Timestamp.now(tz="UTC").isoformat(),
        **{name: version(name) for name in ["numpy", "pandas", "scikit-learn", "xgboost", "torch"]},
    }

## 2. Dataset Loading and Integrity Verification

`market_data` holds identifiers, the target and the few characteristics used for universes and diagnostics. It is loaded only when a
stage needs it. Predictors are read later, one year at a time. The source Parquet files are never modified.

In [ ]:
MARKET_COLUMNS = METADATA + ["ret_1_0", "prc", "dolvol_126d"]


def load_market_data():
    frames = {}
    for country, path in FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset: {path}. See README.md.")
        df = pd.read_parquet(path, columns=MARKET_COLUMNS)
        df["eom"] = pd.to_datetime(df["eom"])
        for column in ["excntry", "gvkey", "size_grp"]:
            df[column] = df[column].astype("category")
        df = df.sort_values(KEYS, ignore_index=True)
        df["predicted_month"] = df["eom"] + pd.offsets.MonthEnd(1)
        df["investable"] = investable_rule(df)
        frames[country] = df
    return frames


def investable_rule(df):
    """Size and liquidity rule, known at portfolio formation (see Section 4)."""
    size = df["size_grp"].astype("string").isin(["small", "large", "mega"])
    return size & df["prc"].abs().ge(1.0) & df["dolvol_126d"].gt(0)


market_data = load_market_data() if ANY_STAGE else {}
DATA_FINGERPRINTS = {country: dataset_fingerprint(path) for country, path in FILES.items()} if ANY_STAGE else {}

In [ ]:
def check_integrity(frames):
    """Counts that must be zero, from the loaded columns and the Parquet schemas."""
    calendar = pd.date_range("2000-01-31", max(df["eom"].max() for df in frames.values()), freq="ME")
    schemas = {country: pq.read_schema(FILES[country]) for country in frames}
    table = pd.DataFrame({country: {
        "Wrong country code": (df["excntry"] != country).sum(),
        "Missing id or eom": df[KEYS].isna().any(axis=1).sum(),
        "Duplicate stock-months": df.duplicated(KEYS).sum(),
        "Not a month-end date": (df["eom"] != df["eom"] + pd.offsets.MonthEnd(0)).sum(),
        "Screen violations": (df[SCREENS] != 1).any(axis=1).sum(),
        "Missing months": len(calendar.difference(df["eom"])),
        "Infinite targets": np.isinf(df[TARGET]).sum(),
        "Unexpected row count": abs(len(df) - EXPECTED_ROWS[country]),
        "File differs from the audited file": int(DATA_FINGERPRINTS[country] != AUDITED_FINGERPRINTS[country]),
        "Schema differs from USA": int(not schemas[country].equals(schemas["USA"])),
    } for country, df in frames.items()})
    display(table)
    if table.to_numpy().sum() > 0:
        raise ValueError("Integrity check failed; see the nonzero counts above.")


if RUN_DATA_CHECKS:
    check_integrity(market_data)

## 3. Target Construction and Temporal Alignment

The signal date is month-end `eom`. The target `ret_exc_lead1m` is the USD return of the **next** month in excess of the U.S. T-bill
rate, so it should equal the next month's `ret_1_0` minus a small rate. Records are assigned to periods by `predicted_month`.
Missing targets are never filled; a mask excludes them from losses and metrics.

In [ ]:
def target_alignment(frames):
    """Compare every target with the realized return of the following month."""
    rows = []
    for country, df in frames.items():
        by_stock = df.groupby("id")
        consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
        rate = (by_stock["ret_1_0"].shift(-1) - df[TARGET])[consecutive].dropna()
        aligned = rate.between(-0.0002, 0.01)
        rows.append({"country": country, "pairs": len(rate), "mismatches": int((~aligned).sum()), "agreement": aligned.mean(),
                     "missing_target": int(df[TARGET].isna().sum())})
    return pd.DataFrame(rows)


if RUN_DATA_CHECKS:
    alignment = target_alignment(market_data)
    display(alignment)
    save_table(alignment, "target_alignment")
    if alignment["agreement"].min() < 0.99:
        raise ValueError("A target is not aligned with the following month.")

## 4. Data-Quality and Outlier Investigation

- **Units.** Returns are decimals (1.0 = 100%). A simple return below −100% is impossible. The excess return may be slightly lower,
  because the T-bill rate is subtracted, so `ret_1_0` is checked.
- **Extreme returns.** For each target above +300%, available price and market-equity growth are compared with the return.
  Agreement suggests a real price change, disagreement a corporate action or data error, and neither is proof.
- **Not checkable here.** The files contain no adjustment factor, delisting return or exchange rate. The cause of an extreme return
  therefore needs the WRDS source tables. The search below confirms which columns exist.
- **Version 2 audit.** All checked targets match the next month's return. Of the 100 largest UK returns per period, 97–99 agree with price growth and
  97–100 are penny stocks. Nano stocks hold 89–100% of UK squared returns, so raw MSE describes a handful of stocks; the MSE rules
  below therefore use clipped MSE as well.
- **Treatment.** Training targets are clipped at quantiles of the training window only. Original returns are used for all evaluation.

**Universe policy, fixed before Version 3 results.** A stock is *investable* if, at formation, its JKP size group is small, large or mega,
its price is at least 1 USD, and its 126-day dollar volume is positive. The $1 and size cut-offs are conventions and are not tuned. The size
part resembles a Version 2 sensitivity whose test results are known, so the comparison is controlled but not blind. The model always
sees the full cross-section; only evaluation and portfolios use the restricted universe.

In [ ]:
def quality_profiles(frames):
    """Return distribution by country and size group through 2019; the test period is not profiled here."""
    rows = []
    for country, df in frames.items():
        part = df[df["predicted_month"] <= VALIDATION[1]]
        total = (part[TARGET] ** 2).sum()
        for size, group in part.groupby(part["size_grp"].astype("string").fillna("unknown")):
            values = group[TARGET].dropna()
            rows.append({
                "country": country, "size_group": size, "stock_months": len(group), "missing_target": int(group[TARGET].isna().sum()),
                "impossible_returns": int((group["ret_1_0"] < -1).sum()), "above_300pct": int((values > 3).sum()),
                "largest": values.max(), "smallest": values.min(), "investable_share": group["investable"].mean(),
                "share_of_squared_returns": (values ** 2).sum() / total,
            })
    return pd.DataFrame(rows)


def extreme_evidence(frames, threshold=3.0):
    """Counts of price evidence for extreme returns, without identifiers."""
    rows = []
    for country, df in frames.items():
        by_stock = df.groupby("id")
        consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
        growth = 1 + by_stock["ret_1_0"].shift(-1).where(consecutive)
        extreme = (df[TARGET] > threshold) & (df["predicted_month"] <= VALIDATION[1])
        row = {"country": country, "extreme_returns": int(extreme.sum()), "next_month_row": int((extreme & consecutive).sum()),
               "penny_stock_at_formation": int((extreme & (df["prc"].abs() < 1)).sum())}
        for name, column in [("price", "prc"), ("market_equity", "me")]:
            ratio = (by_stock[column].shift(-1).abs() / df[column].abs()).where(consecutive)[extreme]
            gap = np.log(ratio / growth[extreme]).abs()
            row[f"{name}_growth_known"], row[f"{name}_growth_agrees"] = int(gap.notna().sum()), int((gap < 0.5).sum())
        rows.append(row)
    return pd.DataFrame(rows)


def adjustment_columns(names):
    words = ["adj", "dlret", "delist", "ajex", "split", "fx", "curcd", "ret_local"]
    return [name for name in names if any(word in name.lower() for word in words)]

## 5. Chronological Train/Validation/Test Separation

| Context | Training ends | Stopping block (early stopping, learning rate, calibration) | Scoring block |
|---|---|---|---|
| `fold1` | Dec 2011 | 2012–2013 | 2014–2015 |
| `fold2` | Dec 2013 | 2014–2015 | 2016–2017 |
| `final` | Dec 2017 | 2018–2019 (official validation) | 2018–2019 |

U.S. training always starts in February 2000. Local windows are the last 12, 36 or 60 predicted months before the training end. In the
folds, the scoring block is never used for fitting, and predictors after 2017 are not loaded. In `final`, validation chooses both checkpoints
and the model, so its scores are optimistic. The two folds overlap in time (`fold2` trains and stops on years that `fold1` scores), so they
are not independent.

The last training return becomes known at the close of the first evaluation formation date, which is when that signal is formed.
This convention is the same as in Versions 1 and 2; `check_chronology` enforces it.

In [ ]:
def month_range(first, last):
    return pd.Timestamp(first) + pd.offsets.MonthEnd(0), pd.Timestamp(last) + pd.offsets.MonthEnd(0)


SOURCE_START = month_range("2000-02", "2000-02")[0]
VALIDATION, TEST = month_range("2018-01", "2019-12"), month_range("2020-01", "2025-12")
CONTEXTS = {
    "fold1": {"train_end": month_range("2011-12", "2011-12")[0], "stop": month_range("2012-01", "2013-12"), "score": month_range("2014-01", "2015-12")},
    "fold2": {"train_end": month_range("2013-12", "2013-12")[0], "stop": month_range("2014-01", "2015-12"), "score": month_range("2016-01", "2017-12")},
    "final": {"train_end": month_range("2017-12", "2017-12")[0], "stop": VALIDATION, "score": VALIDATION},
}
FOLDS = ["fold1", "fold2"]


def train_period(context, window):
    """First and last predicted-return month of a training window."""
    end = CONTEXTS[context]["train_end"]
    if window == "train_full":
        return SOURCE_START, end
    return end - pd.offsets.MonthEnd(WINDOWS[window] - 1), end


def check_chronology(train_months, evaluation_months):
    """Training labels must be realized no later than the first evaluation signal date."""
    last_label = max(month["predicted_month"] for month in train_months)
    first_signal = min(month["eom"] for month in evaluation_months)
    if last_label > first_signal:
        raise ValueError(f"Leakage: a training label of {last_label:%Y-%m} follows the signal date {first_signal:%Y-%m}.")


for name, context in CONTEXTS.items():
    first, last = train_period(name, "train_1y")
    assert len(pd.date_range(first, last, freq="ME")) == 12
    assert context["train_end"] < context["stop"][0] and context["score"][1] < TEST[0]
    assert name == "final" or context["stop"][1] < context["score"][0] <= context["score"][1] < VALIDATION[0]

In [ ]:
if RUN_DATA_CHECKS:
    profiles, evidence = quality_profiles(market_data), extreme_evidence(market_data)
    display(profiles)
    display(evidence)
    save_table(profiles, "return_profiles")
    save_table(evidence, "extreme_evidence")
    print(f"Adjustment or delisting columns in the source: {adjustment_columns(pq.read_schema(FILES['USA']).names) or 'none'}")

    rows = []
    for country, df in market_data.items():
        for period, dates in {"final training": (SOURCE_START, CONTEXTS["final"]["train_end"]), "validation": VALIDATION, "test": TEST}.items():
            part = df[df["predicted_month"].between(*dates)]
            rows.append({"country": country, "period": period, "stock_months": len(part), "labeled": part[TARGET].notna().sum(),
                         "months": part["eom"].nunique(), "expected_months": len(pd.date_range(*dates, freq="ME"))})
    splits = pd.DataFrame(rows)
    display(splits)
    if not (splits["months"] == splits["expected_months"]).all():
        raise ValueError("A period has missing months.")

## 6. Feature Preprocessing and Selection

**Predictors.** A characteristic is kept if it covers at least 50% of labeled observations and varies in each of the seven training windows
ending in December 2017. The result must match the 126 predictors of Versions 1 and 2. This rule uses availability only, never returns, so
using one list in the development folds adds no return information.

**Preprocessing.** Within each country-month, predictors are ranked over all screened stocks and mapped to [−0.5, 0.5]. Missing
values become zero, the neutral point.

**Redundancy filter** (screened as one configuration). Among ranked U.S. predictors, sampled once a year inside the training period, the later
predictor of each pair with absolute correlation above 0.90 is dropped. Labels are not used.

In [ ]:
MIN_COVERAGE, REDUNDANCY = 0.50, 0.90
V2_FEATURES_SHA256 = "b63e196f0f70f7c1c617e04f748203fb0d8d2f938dedf02328c41855d98699f8"  # sorted Version 2 predictor names
FEATURES_FILE = RESULTS_DIR / "features.json"


def read_months(country, first_eom, last_eom, columns):
    """Read selected columns for a range of signal dates, ordered by month and stock."""
    dates = [("eom", ">=", first_eom.date()), ("eom", "<=", last_eom.date())]
    df = pd.read_parquet(FILES[country], columns=KEYS + columns, filters=dates)
    df["eom"] = pd.to_datetime(df["eom"])
    return df.sort_values(["eom", "id"], ignore_index=True)


def sorted_hash(names):
    return hashlib.sha256("\n".join(sorted(names)).encode()).hexdigest()


def choose_features():
    """Coverage and variation in the training windows; returns the kept predictors in source order."""
    names = [name for name in pq.read_schema(FILES["USA"]).names if name not in METADATA]
    if len(names) != 153:
        raise ValueError(f"Expected 153 characteristics, found {len(names)}.")
    keep = pd.Series(True, index=names)
    for country in COUNTRIES:
        windows = ["train_full"] if country == "USA" else list(WINDOWS)
        first, last = train_period("final", windows[-1])
        for start in range(0, len(names), 20):
            batch = names[start:start + 20]
            df = read_months(country, first - pd.offsets.MonthEnd(1), last - pd.offsets.MonthEnd(1), [TARGET] + batch)
            df = df[df[TARGET].notna()]
            for window in windows:
                part = df.loc[df["eom"] + pd.offsets.MonthEnd(1) >= train_period("final", window)[0], batch]
                part = part.where(np.isfinite(part))
                keep[batch] &= (part.notna().mean() >= MIN_COVERAGE) & (part.min() < part.max())
    return keep.index[keep].tolist()


def load_features():
    if not FEATURES_FILE.exists():
        raise FileNotFoundError(f"{FEATURES_FILE} is missing: run the data checks first.")
    names = json.loads(FEATURES_FILE.read_text())
    if sorted_hash(names) != V2_FEATURES_SHA256:
        raise ValueError("The predictor list differs from the verified Version 2 list.")
    return names


if RUN_DATA_CHECKS:
    names = choose_features()
    if sorted_hash(names) != V2_FEATURES_SHA256:
        raise ValueError(f"{len(names)} predictors were selected, but they differ from the Version 2 list.")
    save_json(FEATURES_FILE, names)
    print(f"{len(names)} predictors selected; identical to Version 2.")
features = load_features() if NEEDS_FEATURES else []

In [ ]:
def rank_within_month(values, months):
    """Rank each column within each month, map the ranks to [-0.5, 0.5], and keep missing values missing."""
    values = values.where(np.isfinite(values))
    by_month = values.groupby(months)
    n = by_month.transform("count")
    scaled = (by_month.rank(method="average") - 1) / (n - 1) - 0.5
    return scaled.mask((n == 1) & values.notna(), 0.0)


def load_months(country, first, last, allow_test=False):
    """One dictionary per country-month, for predicted-return months from first to last."""
    if last > VALIDATION[1] and not allow_test:
        raise ValueError("Test months cannot be loaded for training, tuning or selection.")
    month_end, months = pd.offsets.MonthEnd(1), []
    first_eom, last_eom = first - month_end, last - month_end
    for year in range(first_eom.year, last_eom.year + 1):
        start, end = max(first_eom, pd.Timestamp(year, 1, 31)), min(last_eom, pd.Timestamp(year, 12, 31))
        df = read_months(country, start, end, [TARGET] + features)
        X = rank_within_month(df[features], df["eom"]).fillna(0.0).to_numpy(dtype="float32")
        for eom, rows in df.groupby("eom").indices.items():
            returns = df[TARGET].to_numpy()[rows]
            months.append({
                "country": country, "eom": pd.Timestamp(eom), "predicted_month": pd.Timestamp(eom) + month_end,
                "id": df["id"].to_numpy()[rows], "X": torch.from_numpy(X[rows]), "returns": returns, "labeled": np.isfinite(returns),
            })
    expected = len(pd.date_range(first, last, freq="ME"))
    if len(months) != expected:
        raise ValueError(f"{country}: {len(months)} months found, {expected} expected.")
    return months


def select_months(months, first, last):
    return [month for month in months if first <= month["predicted_month"] <= last]


def clip_limits(months):
    """Training-period quantiles of the labeled returns."""
    returns = np.concatenate([month["returns"][month["labeled"]] for month in months])
    return tuple(float(value) for value in np.quantile(returns, CLIP_QUANTILES))


def redundant_columns(source_months):
    """Positions kept by the redundancy filter, from one U.S. training month per year."""
    sample = np.concatenate([month["X"].numpy() for month in source_months[::12]])
    with np.errstate(invalid="ignore"):
        correlation = np.abs(np.nan_to_num(np.corrcoef(sample, rowvar=False)))
    kept = []
    for position in range(correlation.shape[0]):
        if not kept or correlation[position, kept].max() <= REDUNDANCY:
            kept.append(position)
    return kept

## 7. Baseline Model Training

This section first defines what every model shares: metrics, calibration and verified run records.

**Metrics.** All are monthly and then averaged with equal month weights: rank IC (Spearman), Pearson IC, raw MSE against a zero forecast, and
clipped MSE, where realized returns are clipped at the **destination country's training-window** quantiles for every model.
Raw returns stay unchanged in all other metrics.

**Calibration.** Rank-oriented models order stocks well, but the size of their forecasts is uncertain. The calibrated
forecast is `slope × within-month z-score`. The slope (return per standard deviation) is the least-squares slope on clipped, demeaned returns
of the stopping block, limited to [0.0005, 0.03]. It is positive, so **ranks and IC never change**, and the market-wide monthly return,
which the predictors do not forecast, is set to zero. In the folds the scoring block is out of sample. For 2018–2019 the slope of one year is
applied to the other year, which is not chronological and affects validation MSE only; test forecasts use the slope of both years. Because checkpoints
are chosen on the same block, the slope may be slightly too large.

In [ ]:
SLOPE_LIMITS = (0.0005, 0.03)


def correlation(a, b):
    a, b = a - a.mean(), b - b.mean()
    scale = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / scale) if scale > 0 and np.ptp(a) > 0 and np.ptp(b) > 0 else np.nan


def rank_ic(prediction, actual):
    """Spearman correlation across stocks, with average ranks for ties."""
    return correlation(pd.Series(prediction).rank().to_numpy(), pd.Series(actual).rank().to_numpy())


def zscore(table):
    """Within-month standardized forecasts over all stocks of the month; zero if all are equal."""
    by_month = table.groupby("eom")["prediction"]
    spread = by_month.transform("std", ddof=0)
    z = (table["prediction"] - by_month.transform("mean")) / spread.where(spread > 1e-12)
    return z.fillna(0.0)


def calibration_slope(table, limits):
    """Monthly-equal least-squares slope of clipped, demeaned returns on standardized forecasts."""
    part = pd.DataFrame({"eom": table["eom"], "z": zscore(table), "y": table[TARGET].clip(*limits)}).dropna()
    part["y"] -= part.groupby("eom")["y"].transform("mean")
    part["z"] -= part.groupby("eom")["z"].transform("mean")
    covariance = (part["z"] * part["y"]).groupby(part["eom"]).mean().mean()
    variance = (part["z"] ** 2).groupby(part["eom"]).mean().mean()
    slope = covariance / variance if variance > 0 else SLOPE_LIMITS[0]
    return float(np.clip(slope, *SLOPE_LIMITS))


def calibrate(fit, apply, limits):
    return apply.assign(calibrated=calibration_slope(fit, limits) * zscore(apply))


def calibrate_validation(table, limits):
    """Cross-fitted by year: each year uses the slope of the other years."""
    year = (table["eom"] + pd.offsets.MonthEnd(1)).dt.year
    parts = [calibrate(table[year != y], table[year == y], limits) for y in sorted(year.unique())]
    return pd.concat(parts)


def monthly_scores(table, limits):
    """Metrics by month for the full and the investable universe."""
    rows = []
    labeled = table[table[TARGET].notna()]
    for universe, part in [("full", labeled), ("investable", labeled[labeled["investable"]])]:
        for eom, month in part.groupby("eom"):
            y, p, c = (month[name].to_numpy(dtype=float) for name in [TARGET, "prediction", "calibrated"])
            clipped = np.clip(y, *limits)
            rows.append({
                "universe": universe, "predicted_month": eom + pd.offsets.MonthEnd(1), "stocks": len(y), "rank_ic": rank_ic(p, y),
                "pearson_ic": correlation(p, y), "mse": np.mean((y - p) ** 2), "mse_calibrated": np.mean((y - c) ** 2),
                "mse_zero": np.mean(y ** 2), "clipped_mse_calibrated": np.mean((clipped - c) ** 2), "clipped_mse_zero": np.mean(clipped ** 2),
            })
    return pd.DataFrame(rows)


def summarize_scores(monthly):
    """Average monthly scores of one forecast and universe; an undefined month makes the IC undefined."""
    means = monthly[["rank_ic", "pearson_ic", "mse", "mse_calibrated", "mse_zero", "clipped_mse_calibrated", "clipped_mse_zero"]].mean(skipna=False)
    return {
        **means.to_dict(), "months": len(monthly), "ic_std": monthly["rank_ic"].std(), "undefined_ic_months": int(monthly["rank_ic"].isna().sum()),
        "mse_gain_raw": 1 - means["mse"] / means["mse_zero"], "mse_gain_calibrated": 1 - means["mse_calibrated"] / means["mse_zero"],
        "clipped_gain_calibrated": 1 - means["clipped_mse_calibrated"] / means["clipped_mse_zero"],
    }


def add_actuals(table, country):
    """Add realized returns and universe membership; one row per stock-month."""
    actual = market_data[country][KEYS + [TARGET, "investable"]]
    merged = table.merge(actual, on=KEYS, how="left", validate="one_to_one", indicator=True)
    if (merged["_merge"] != "both").any():
        raise ValueError(f"{country}: a forecast has no matching stock-month.")
    return merged.drop(columns="_merge")


def local_limits(context, country, window):
    """Clipping bounds of the destination country's training window."""
    df = market_data[country]
    returns = df.loc[df["predicted_month"].between(*train_period(context, window)), TARGET].dropna()
    return tuple(float(value) for value in np.quantile(returns, CLIP_QUANTILES))

**Run records.** Every trained model is one run with a JSON record, a model file and, for local markets, a forecast file. A run is complete only if
its record has the current fingerprint and both checksums match. Incomplete runs are trained again; a record from other settings raises
an error instead of being overwritten. Runs are short (minutes), so an interrupted job resumes at the first missing run.

In [ ]:
def run_path(context, config, strategy, country, window, seed, suffix):
    return RUNS_DIR / context / config / f"{strategy}_{country}_{window}_s{seed}{suffix}"


def run_complete(key, fingerprint):
    """True if the run is recorded with this fingerprint and its files are intact."""
    path = run_path(*key, ".json")
    if not path.exists():
        return False
    record = json.loads(path.read_text())
    if record["fingerprint"] != fingerprint:
        raise ValueError(f"{path} was produced with other settings. Move the results folder before rerunning.")
    return all((path.parent / name).exists() and file_sha256(path.parent / name) == digest for name, digest in record["files"].items())


def load_record(key):
    path = run_path(*key, ".json")
    if not path.exists():
        raise FileNotFoundError(f"Missing run: {path}")
    record = json.loads(path.read_text())
    for name, digest in record["files"].items():
        if file_sha256(path.parent / name) != digest:
            raise ValueError(f"Damaged file: {path.parent / name}")
    return record


def finish_run(key, fingerprint, details, model_suffix=None, forecasts=None):
    """Save the forecasts, then the record; the model file is already saved."""
    files = {}
    if model_suffix:
        path = run_path(*key, model_suffix)
        files[path.name] = file_sha256(path)
    if forecasts is not None:
        path = run_path(*key, ".parquet")
        save_safely(path, lambda temporary: forecasts.to_parquet(temporary, index=False))
        files[path.name] = file_sha256(path)
    record = {"key": list(key), "fingerprint": fingerprint, **details, "files": files, "environment": environment_details()}
    save_json(run_path(*key, ".json"), record)
    return record


def load_forecast(key, block):
    load_record(key)  # verifies the checksum
    table = pd.read_parquet(run_path(*key, ".parquet"), filters=[("block", "==", block)])
    return table.drop(columns="block")


def forecast_table(months, predictions, block):
    return pd.concat([
        pd.DataFrame({"block": block, "id": month["id"], "eom": month["eom"], "prediction": prediction.astype("float64")})
        for month, prediction in zip(months, predictions)
    ], ignore_index=True)


def base_fingerprint(key):
    """Everything common to all runs that determines a result."""
    context = key[0]
    return {
        "code": CODE_VERSION, "key": list(key), "features": short_hash(features), "data": DATA_FINGERPRINTS,
        "context": {name: str(value) for name, value in CONTEXTS[context].items()}, "clip": CLIP_QUANTILES, "epochs": MAX_EPOCHS,
    }

**Baselines.** Elastic Net and XGBoost (pseudo-Huber, the best Version 2 baseline) are trained on clipped returns relative to the month's
median. A second XGBoost is trained on return ranks, so the Transformers are not the only models with a ranking target. Every month has
equal weight. The setting with the highest validation rank IC is kept, the same criterion as for the Transformers.

In [ ]:
BASELINE_GRIDS = {
    "elastic_net": [{"alpha_share": a, "l1_ratio": r} for a, r in product([0.3, 0.1, 0.03, 0.01], [0.1, 0.5])],
    "xgboost": [{"max_depth": d, "n_estimators": n, "min_child_weight": w} for d, n, w in product([2, 4], [100, 300], [20, 200])],
}
BASELINE_GRIDS["xgboost_rank"] = BASELINE_GRIDS["xgboost"]
BASELINES = list(BASELINE_GRIDS)


def stack_baseline(months, limits, ranked):
    """Labeled rows, targets and month weights (average one)."""
    X, y, weight = [], [], []
    for month in months:
        labeled = month["labeled"]
        returns = month["returns"][labeled]
        if len(returns) < 2:
            continue
        target = np.clip(returns, *limits) - np.median(np.clip(returns, *limits))
        if ranked:
            target = (pd.Series(returns).rank().to_numpy() - 1) / (len(returns) - 1) - 0.5
        X.append(month["X"].numpy()[labeled])
        y.append(target)
        weight.append(np.full(len(returns), 1 / len(returns)))
    weight = np.concatenate(weight)
    return np.concatenate(X), np.concatenate(y), weight / weight.mean()


def make_baseline(family, params, X, y, weight):
    if family == "elastic_net":
        centered = X - np.average(X, axis=0, weights=weight)
        alpha_max = np.abs(centered.T @ (weight * (y - np.average(y, weights=weight)))).max() / len(y) / params["l1_ratio"]
        return ElasticNet(alpha=params["alpha_share"] * alpha_max, l1_ratio=params["l1_ratio"], precompute=True, max_iter=100_000, random_state=SEED)
    return XGBRegressor(
        **params, objective="reg:squarederror" if family == "xgboost_rank" else "reg:pseudohubererror", huber_slope=0.10, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.5, reg_lambda=10.0, base_score=0.0, tree_method="hist", n_jobs=N_JOBS, random_state=SEED,
    )


def months_ic(months, predictions):
    """Average monthly rank IC; minus infinity if a month has no defined ranking."""
    values = [rank_ic(p[m["labeled"]], m["returns"][m["labeled"]]) for m, p in zip(months, predictions) if m["labeled"].sum() > 1]
    value = float(np.mean(values)) if values else np.nan
    return value if np.isfinite(value) else -np.inf


def run_baseline(family, country, window, train_months, stop_months):
    key = ("final", "baselines", family, country, window, SEED)
    fingerprint = short_hash({**base_fingerprint(key), "grid": BASELINE_GRIDS[family]})
    if run_complete(key, fingerprint):
        print(f"{' | '.join(map(str, key))}: complete")
        return
    check_chronology(train_months, stop_months)
    started = time.time()
    X, y, weight = stack_baseline(train_months, clip_limits(train_months), family == "xgboost_rank")
    best, trials = None, []
    for params in BASELINE_GRIDS[family]:
        model = make_baseline(family, params, X, y, weight)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(X, y, sample_weight=weight)
        converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
        predictions = [np.asarray(model.predict(month["X"].numpy())) for month in stop_months]
        value = months_ic(stop_months, predictions)
        trials.append({**params, "stop_ic": value if np.isfinite(value) else None, "converged": converged})
        if best is None or value > best[0]:
            best = (value, model, predictions)
    path = run_path(*key, ".joblib")
    save_safely(path, lambda temporary: joblib.dump(best[1], temporary))
    finish_run(key, fingerprint, {"trials": trials, "stop_ic": best[0] if np.isfinite(best[0]) else None, "n_parameters": 0, "seconds": time.time() - started},
               ".joblib", forecast_table(stop_months, best[2], "score"))
    print(f"{' | '.join(map(str, key))}: validation IC {best[0]:.4f}, {time.time() - started:.0f}s", flush=True)


if RUN_BASELINES:
    for country in LOCAL_COUNTRIES:
        months = load_months(country, train_period("final", "train_5y")[0], VALIDATION[1])
        for window, family in product(WINDOWS, BASELINES):
            run_baseline(family, country, window, select_months(months, *train_period("final", window)), select_months(months, *VALIDATION))

## 8. Transformer Architecture

The input is one country-month: a matrix of stocks by predictors. Attention runs across **all** stocks of that month, including unlabeled
ones; cross-sections are never split. Training uses one optimizer step per month, so every month has equal weight whatever its size. Memory-efficient
attention kernels compute the same full attention.

- `full`: pre-normalized Transformer blocks (attention, then feed-forward), as in Version 2.
- `gated`: a stockwise residual network whose attention is scaled by a learned gate starting at zero. It begins as a stock-by-stock model and
  adds interactions only if they help. This is a different, simpler architecture and is labeled as such.

In [ ]:
HEADS, FEEDFORWARD = 4, 128


class Attention(nn.Module):
    """Pre-normalized attention across the stocks of one month."""

    def __init__(self, dim, dropout, gated):
        super().__init__()
        self.dropout_rate = dropout
        self.norm = nn.LayerNorm(dim)
        self.query_key_value = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Parameter(torch.zeros(dim)) if gated else None

    def forward(self, x):
        n_stocks, dim = x.shape
        query, key, value = [
            part.view(1, n_stocks, HEADS, dim // HEADS).transpose(1, 2) for part in self.query_key_value(self.norm(x)).chunk(3, dim=-1)
        ]
        attended = F.scaled_dot_product_attention(query, key, value, dropout_p=self.dropout_rate if self.training else 0.0)
        attended = self.dropout(self.output(attended.transpose(1, 2).reshape(n_stocks, dim)))
        return x + (attended if self.gate is None else torch.tanh(self.gate) * attended)


class FeedForward(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.Sequential(
            nn.Linear(dim, FEEDFORWARD), nn.ReLU(), nn.Dropout(dropout), nn.Linear(FEEDFORWARD, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.layers(self.norm(x))


class CrossSectionalModel(nn.Module):
    """One forecast per stock from a (stocks, predictors) matrix of a single country-month."""

    def __init__(self, columns, dim, n_blocks, dropout, gated):
        super().__init__()
        self.register_buffer("columns", torch.as_tensor(columns, dtype=torch.long))  # predictor positions used
        self.input_layer = nn.Linear(len(columns), dim)
        self.input_dropout = nn.Dropout(dropout)
        order = (lambda a, f: nn.Sequential(f, a)) if gated else (lambda a, f: nn.Sequential(a, f))
        self.blocks = nn.ModuleList([order(Attention(dim, dropout, gated), FeedForward(dim, dropout)) for _ in range(n_blocks)])
        self.output_norm = nn.LayerNorm(dim)
        self.output_layer = nn.Linear(dim, 1)

    def forward(self, x):
        x = self.input_dropout(self.input_layer(x[:, self.columns]))
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.output_norm(x)).squeeze(-1)


def select_device(choice):
    available = {"cuda": torch.cuda.is_available(), "mps": torch.backends.mps.is_available(), "cpu": True}
    choice = next(name for name, usable in available.items() if usable) if choice == "auto" else choice
    if not available.get(choice, False):
        raise ValueError(f"Device '{choice}' is not available.")
    return torch.device(choice)


DEVICE = select_device(DEVICE_CHOICE)


def make_model(spec, columns):
    return CrossSectionalModel(columns, spec["dim"], spec["blocks"], spec["dropout"], spec["gated"]).to(DEVICE)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

**Loss.** Each month contributes a weighted sum of at most three terms, computed on labeled stocks only:

| Term | Definition | Scale |
|---|---|---|
| Regression | 2 × Huber loss (delta 0.10), divided by its average value for a zero forecast on the training months | 1 at a zero forecast |
| Rank | 1 − Pearson correlation of forecasts and return ranks | 1 for uncorrelated forecasts, 0 at best |
| Magnitude | 1 − Pearson correlation of forecasts and clipped returns | same |

The regression target is the clipped return (`raw`) or the clipped return minus the month's median (`relative`), which removes the market
movement that predictors cannot order. The correlation terms cost O(N log N) for ranks computed once, not O(N²) pairs. They ignore the
forecast scale, so the regression term alone sets it; their gradient is proportional to one over the forecast spread, and the
constant `1e-8` keeps it finite for constant forecasts.

In [ ]:
def prepare_targets(months, spec, limits):
    """Tensors of one training month; `limits` maps a country to its training-period clipping bounds."""
    prepared = []
    for month in months:
        labeled = month["labeled"]
        if labeled.sum() < 2:
            continue
        returns = month["returns"][labeled]
        clipped = np.clip(returns, *limits[month["country"]])
        ranks = pd.Series(returns).rank().to_numpy()
        target = clipped - np.median(clipped) if spec["target"] == "relative" else clipped
        prepared.append({
            "X": month["X"], "labeled": torch.from_numpy(np.flatnonzero(labeled)), "target": torch.tensor(target, dtype=torch.float32),
            "ranks": unit_vector(ranks), "magnitudes": unit_vector(clipped), "name": f"{month['country']} {month['eom']:%Y-%m}",
        })
    if not prepared:
        raise ValueError("No labeled training months.")
    return prepared


def unit_vector(values):
    centered = values - values.mean()
    return torch.tensor(centered / max(np.sqrt((centered ** 2).sum()), 1e-12), dtype=torch.float32)


def regression_loss(prediction, target):
    return 2 * F.huber_loss(prediction, target, delta=0.10)


def correlation_loss(prediction, unit_target):
    centered = prediction - prediction.mean()
    return 1 - (centered * unit_target).sum() / torch.sqrt((centered ** 2).sum() + 1e-8)


def zero_forecast_loss(prepared):
    return float(np.mean([regression_loss(torch.zeros_like(m["target"]), m["target"]).item() for m in prepared]))


def month_loss(prediction, month, spec, scale):
    """Loss of one prepared month from the forecasts of its labeled stocks."""
    loss = spec["regression_weight"] * regression_loss(prediction, month["target"].to(prediction.device)) / scale
    if spec["rank_weight"]:
        loss = loss + spec["rank_weight"] * correlation_loss(prediction, month["ranks"].to(prediction.device))
    if spec["magnitude_weight"]:
        loss = loss + spec["magnitude_weight"] * correlation_loss(prediction, month["magnitudes"].to(prediction.device))
    return loss


def predict_months(model, months):
    """Forecasts of all stocks, one array per month."""
    model.eval()
    with torch.inference_mode():
        return [model(month["X"].to(DEVICE)).cpu().numpy() for month in months]

**Training.** AdamW with one step per month. Checkpoints are selected by the average rank IC of the stopping block, because raw MSE is dominated by
a few extreme returns. Training stops after 10 epochs without improvement. The optional `stable` recipe adds gradient clipping, a 50-step
warm-up, learning-rate halving after 4 stalled epochs and an exponential moving average of the weights, which is the model that is
validated and saved. When training starts from a pretrained model, that model itself is epoch 0, so fine-tuning is kept only if it helps.

In [ ]:
PATIENCE, MIN_IMPROVEMENT, WARMUP_STEPS, STALL_EPOCHS, HEAD_EPOCHS = 10, 1e-4, 50, 4, 3


def parameter_groups(model, lr, staged):
    """One group for all weights, or layer-wise learning rates (staged fine-tuning)."""
    if not staged:
        return [{"params": list(model.parameters()), "scale": 1.0, "head": True}]
    n = len(model.blocks)
    layers = [list(model.output_layer.parameters()) + list(model.output_norm.parameters())]
    layers += [list(block.parameters()) for block in reversed(model.blocks)] + [list(model.input_layer.parameters())]
    assert sum(len(layer) for layer in layers) == len(list(model.parameters())) and len(layers) == n + 2
    return [{"params": layer, "scale": 0.5 ** depth, "head": depth == 0} for depth, layer in enumerate(layers)]


def copy_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def train_model(prepared, stop_months, spec, columns, seed, lr, label, initial_state=None, staged=False):
    """Train one model with early stopping; returns the best weights and a record."""
    torch.manual_seed(seed)  # also seeds CUDA and MPS
    shuffler = np.random.default_rng(seed)
    model = make_model(spec, columns)
    if initial_state is not None:
        model.load_state_dict(initial_state)
    average = make_model(spec, columns) if spec["stable"] else model
    average.load_state_dict(model.state_dict())
    groups = parameter_groups(model, lr, staged)
    optimizer = torch.optim.AdamW([{**group, "lr": 0.0} for group in groups], weight_decay=spec["weight_decay"])
    scale, step, factor, started = zero_forecast_loss(prepared), 0, 1.0, time.time()

    best_ic, best_epoch, best_state, history, stalled = -np.inf, 0, None, [], 0
    if initial_state is not None:
        best_ic, best_state = months_ic(stop_months, predict_months(average, stop_months)), copy_state(average)
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for i in shuffler.permutation(len(prepared)):
            month, step = prepared[i], step + 1
            warmup = min(1.0, step / WARMUP_STEPS) if spec["stable"] else 1.0
            for group in optimizer.param_groups:
                frozen = staged and epoch <= HEAD_EPOCHS and not group["head"]
                group["lr"] = 0.0 if frozen else lr * group["scale"] * warmup * factor
            optimizer.zero_grad(set_to_none=True)
            prediction = model(month["X"].to(DEVICE))[month["labeled"].to(DEVICE)]
            loss = month_loss(prediction, month, spec, scale)
            loss.backward()
            norm = nn.utils.clip_grad_norm_(model.parameters(), 1.0 if spec["stable"] else float("inf"), error_if_nonfinite=False)
            if not (torch.isfinite(loss) and torch.isfinite(norm)):
                raise FloatingPointError(f"{label}: nonfinite loss or gradient in {month['name']}.")
            optimizer.step()
            if spec["stable"]:
                decay = min(0.99, (1 + step) / (10 + step))
                with torch.no_grad():
                    for target, source in zip(average.parameters(), model.parameters()):
                        target.lerp_(source, 1 - decay)
            losses.append(loss.item())
        value = months_ic(stop_months, predict_months(average, stop_months))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "stop_ic": value if np.isfinite(value) else None})
        if value > best_ic + MIN_IMPROVEMENT or best_state is None:
            best_ic, best_epoch, best_state, stalled = value, epoch, copy_state(average), 0
        else:
            stalled += 1
        if spec["stable"] and stalled and stalled % STALL_EPOCHS == 0:
            factor *= 0.5
        if epoch - best_epoch >= PATIENCE + (HEAD_EPOCHS if staged else 0):
            break
    if not np.isfinite(best_ic):
        raise FloatingPointError(f"{label}: no defined rank IC in the stopping block.")
    print(f"  {label} | lr {lr:g} | best epoch {best_epoch}/{epoch} | stop IC {best_ic:.4f} | {time.time() - started:.0f}s", flush=True)
    return best_state, {"lr": lr, "best_epoch": best_epoch, "epochs": epoch, "stop_ic": best_ic, "history": history,
                        "n_parameters": count_parameters(model), "seconds": time.time() - started}

## 9. U.S. Full-History Pretraining

`pretrained` uses every U.S. month from February 2000 to the end of the context's training period; local windows never shorten it. It is
trained once per configuration, context and seed, and the verified checkpoint is reused by `direct`, `finetuned` and `staged`.
The learning rate (1e-3 or 3e-4) is chosen with the first seed on the U.S. stopping block and kept for the other seeds.

## 10. Target-Market Local Model Training

`local` trains the same architecture from random weights on 12, 36 or 60 months of one country.

## 11. Transfer Learning and Fine-Tuning

| Strategy | Training |
|---|---|
| `direct` | pretrained U.S. model, unchanged |
| `finetuned` | all weights updated from the U.S. model; learning rate 3e-4 or 1e-4 |
| `staged` | the output layer alone for 3 epochs, then all layers with learning rates halved at each deeper layer; same two rates |
| `joint` | one model on all U.S. and local months together |

One function trains every strategy.

In [ ]:
STRATEGIES = ["direct", "local", "finetuned", "staged", "joint"]
LEARNING_RATES = {"pretrained": [1e-3, 3e-4], "local": [1e-3, 3e-4], "joint": [1e-3, 3e-4], "finetuned": [3e-4, 1e-4], "staged": [3e-4, 1e-4]}


def load_checkpoint(key):
    """Verified weights of a run (tensors and plain values only)."""
    load_record(key)
    saved = torch.load(run_path(*key, ".pt"), map_location="cpu", weights_only=True)
    if saved["key"] != list(key) or saved["features"] != features:
        raise ValueError(f"{run_path(*key, '.pt')}: the checkpoint belongs to other settings.")
    return saved


def save_tensors(path, payload):
    """Save through a file handle: the file then has the same checksum whatever its temporary name."""
    with open(path, "wb") as file:
        torch.save(payload, file)


def source_key(key):
    context, config, _, _, _, seed = key
    return (context, config, "pretrained", "USA", "train_full", seed)


class MonthStore:
    """Months of one context, loaded once per country and never beyond its scoring block."""

    def __init__(self, context):
        self.context, self.cache = context, {}

    def get(self, country, name):
        if country not in self.cache:
            first = train_period(self.context, "train_full" if country == "USA" else "train_5y")[0]
            self.cache[country] = load_months(country, first, CONTEXTS[self.context]["score"][1])
        dates = CONTEXTS[self.context][name] if name in ["stop", "score"] else train_period(self.context, name)
        return select_months(self.cache[country], *dates)

    def columns(self, spec):
        return redundant_columns(self.get("USA", "train_full")) if spec["filtered"] else list(range(len(features)))


def run_fingerprint(key, spec, source_sha=None):
    return short_hash({**base_fingerprint(key), "spec": spec, "rates": LEARNING_RATES.get(key[2]), "source": source_sha,
                       "training": [PATIENCE, MIN_IMPROVEMENT, WARMUP_STEPS, STALL_EPOCHS, HEAD_EPOCHS, HEADS, FEEDFORWARD, REDUNDANCY]})


def run_model(key, spec, store):
    """Train or reuse one model, forecast the stopping and scoring blocks, and save a verified record."""
    context, config, strategy, country, window, seed = key
    uses_source = strategy in ["direct", "finetuned", "staged"]
    source = load_record(source_key(key)) if uses_source else None
    source_sha = source["files"][run_path(*source_key(key), ".pt").name] if uses_source else None
    fingerprint = run_fingerprint(key, spec, source_sha)
    label = " | ".join(map(str, key))
    if run_complete(key, fingerprint):
        print(f"{label}: complete", flush=True)
        return
    stop_months, score_months = store.get(country, "stop"), store.get(country, "score")
    columns = load_checkpoint(source_key(key))["columns"] if uses_source else store.columns(spec)
    initial_state = load_checkpoint(source_key(key))["state"] if uses_source else None

    if strategy == "direct":
        state, details = initial_state, {"n_parameters": source["n_parameters"], "seconds": 0.0, "best_epoch": source["best_epoch"], "lr": source["lr"]}
    else:
        train_months = {country: store.get(country, window)}
        if strategy == "joint":
            train_months["USA"] = store.get("USA", "train_full")
        for months in train_months.values():
            check_chronology(months, stop_months)
        limits = {name: clip_limits(months) for name, months in train_months.items()}
        prepared = prepare_targets([month for months in train_months.values() for month in months], spec, limits)

        first_seed = (context, config, strategy, country, window, SEED)
        rates = LEARNING_RATES[strategy] if seed == SEED else [load_record(first_seed)["lr"]]
        best, trials, failures = None, [], []
        for lr in rates:
            try:
                state, details = train_model(prepared, stop_months, spec, columns, seed, lr, label, initial_state, strategy == "staged")
            except FloatingPointError as error:  # unstable setting: record it and continue
                failures.append(f"lr {lr:g}: {error}")
                continue
            trials.append({"lr": lr, "stop_ic": details["stop_ic"], "best_epoch": details["best_epoch"], "seconds": details["seconds"]})
            if best is None or details["stop_ic"] > best[1]["stop_ic"]:
                best = (state, details)
        if best is None:
            raise RuntimeError(f"{label}: every learning rate was numerically unstable. {failures}")
        state, details = best
        details.update(trials=trials, failures=failures, total_seconds=sum(trial["seconds"] for trial in trials))
        payload = {"key": list(key), "features": features, "columns": columns, "spec": spec, "state": state}
        save_safely(run_path(*key, ".pt"), lambda temporary: save_tensors(temporary, payload))

    forecasts = None
    if country != "USA":
        model = make_model(spec, columns)
        model.load_state_dict(state)
        blocks = {"score": score_months} if context == "final" else {"stop": stop_months, "score": score_months}
        forecasts = pd.concat([forecast_table(months, predict_months(model, months), block) for block, months in blocks.items()], ignore_index=True)
        if not np.isfinite(forecasts["prediction"]).all():
            raise FloatingPointError(f"{label}: nonfinite forecasts.")
    finish_run(key, fingerprint, {"spec": spec, **details}, None if strategy == "direct" else ".pt", forecasts)


def run_configuration(context, config, seeds):
    """All strategies of one configuration in one context, in dependency order."""
    store, spec = MonthStore(context), CONFIGS[config]
    for seed in seeds:
        run_model((context, config, "pretrained", "USA", "train_full", seed), spec, store)
        for strategy, country, window in product(STRATEGIES, LOCAL_COUNTRIES, WINDOWS):
            run_model((context, config, strategy, country, window, seed), spec, store)

## 12. Systematic Hyperparameter Screening

Eight configurations, fixed before any Version 3 result. All but `combined` change one thing relative to `control`, so this is not a full
Cartesian product.

| Configuration | Change from `control` | Why |
|---|---|---|
| `control` | – | Version 2 `rank` objective and architecture, with Version 3 checkpoint selection and learning rates |
| `relative` | month-relative regression target | removes market-wide noise from the regression term |
| `hybrid` | relative target; rank and magnitude correlation with weight 0.5 each | balances order and size |
| `stable` | gradient clipping, warm-up, learning-rate halving, averaged weights, dropout 0.20 | seed variability in Version 2 was larger than model differences |
| `small` | 32 dimensions, 1 block | prefer a smaller model if it performs equally |
| `gated` | gated attention | gated and stockwise models were among the most stable in Version 2 |
| `filtered` | redundant predictors removed | fewer, less correlated inputs |
| `combined` | hybrid + stable + gated | tests whether the changes add up |

**Screening score.** For every configuration and transfer strategy: rank IC averaged over the two scoring blocks, both countries, three windows
and two seeds. A configuration's score is its best strategy among `finetuned`, `staged` and `joint`. Taking the best of three is mildly
optimistic, but equally so for each configuration. The two best configurations become finalists, plus `control` as the reference. Configurations
within 0.001 of each other are ordered by parameter count. A strategy whose calibrated raw or clipped MSE is more than 0.25% worse than a
zero forecast in any country cannot make its configuration a finalist (the reference is always kept). Every configuration has the same runs, seeds and epoch limit.

In [ ]:
CONTROL = {
    "target": "raw", "regression_weight": 1.0, "rank_weight": 1.0, "magnitude_weight": 0.0, "dim": 64, "blocks": 2, "dropout": 0.10,
    "gated": False, "filtered": False, "stable": False, "weight_decay": 1e-2,
}
HYBRID = {"target": "relative", "rank_weight": 0.5, "magnitude_weight": 0.5}
STABLE = {"stable": True, "dropout": 0.20}
CONFIGS = {
    "control": {}, "relative": {"target": "relative"}, "hybrid": HYBRID, "stable": STABLE, "small": {"dim": 32, "blocks": 1},
    "gated": {"gated": True}, "filtered": {"filtered": True}, "combined": {**HYBRID, **STABLE, "gated": True},
}
CONFIGS = {name: {**CONTROL, **changes} for name, changes in CONFIGS.items()}
if not set(ONLY_CONFIGS) <= set(CONFIGS):
    raise ValueError(f"Unknown configurations: {set(ONLY_CONFIGS) - set(CONFIGS)}")
TRANSFER = ["finetuned", "staged", "joint"]
FINALISTS_FILE = RESULTS_DIR / "finalists.json"

if RUN_SCREENING:
    for config, context in product(ONLY_CONFIGS or CONFIGS, FOLDS):
        run_configuration(context, config, SCREENING_SEEDS)

In [ ]:
def equivalent_order(table, score, *ties):
    """Order by score; repeatedly take the simplest candidate among those equivalent to the current leader."""
    remaining, order = table.sort_values([score, "n_parameters"], ascending=[False, True]), []
    while len(remaining):
        close = remaining[remaining[score] >= remaining[score].iloc[0] - EQUIVALENT_IC]
        pick = close.sort_values([*ties, score], ascending=[*[True] * len(ties), False]).index[0]
        order.append(pick)
        remaining = remaining.drop(pick)
    return table.loc[order].reset_index(drop=True)


def screening_scores():
    """One row per run and universe, scored on the scoring block with a slope from the stopping block."""
    rows = []
    for context, config, strategy, country, window, seed in product(FOLDS, CONFIGS, STRATEGIES, LOCAL_COUNTRIES, WINDOWS, SCREENING_SEEDS):
        key = (context, config, strategy, country, window, seed)
        record, limits = load_record(key), local_limits(context, country, window)
        stop, score = (add_actuals(load_forecast(key, block), country) for block in ["stop", "score"])
        monthly = monthly_scores(calibrate(stop, score, limits), limits)
        for universe, part in monthly.groupby("universe"):
            rows.append({"context": context, "config": config, "strategy": strategy, "country": country, "window": window, "seed": seed,
                         "universe": universe, **summarize_scores(part), "n_parameters": record["n_parameters"],
                         "seconds": record.get("total_seconds", record["seconds"])})
    return pd.DataFrame(rows)


def choose_finalists(scores):
    """Ranked configurations and the finalists (two best eligible, plus the reference)."""
    full = scores[scores["universe"] == "full"]
    table = full.groupby(["config", "strategy"]).agg(
        rank_ic=("rank_ic", lambda values: values.mean(skipna=False)), pearson_ic=("pearson_ic", "mean"), mse_gain_calibrated=("mse_gain_calibrated", "mean"),
        n_parameters=("n_parameters", "max"), seconds=("seconds", "sum"),
    ).reset_index()
    gains = full.groupby(["config", "strategy", "country"])[["mse_gain_calibrated", "clipped_gain_calibrated"]].mean()
    worst = gains.min(axis=1).groupby(["config", "strategy"]).min()
    table["worst_country_mse_gain"] = [worst[c, s] for c, s in zip(table["config"], table["strategy"])]
    for column, value in [*product(["context"], FOLDS), *product(["country"], LOCAL_COUNTRIES), *product(["window"], WINDOWS)]:
        part = full[full[column] == value].groupby(["config", "strategy"])["rank_ic"].mean()
        table[f"ic_{value}"] = [part[c, s] for c, s in zip(table["config"], table["strategy"])]
    seed_means = full.groupby(["config", "strategy", "seed"])["rank_ic"].mean().groupby(["config", "strategy"]).std()
    table["seed_std"] = [seed_means[c, s] for c, s in zip(table["config"], table["strategy"])]
    table["eligible"] = (table["worst_country_mse_gain"] >= -MSE_TOLERANCE) & table["rank_ic"].notna() & table["strategy"].isin(TRANSFER)
    eligible = table[table["eligible"]]
    best = eligible.loc[eligible.groupby("config")["rank_ic"].idxmax()]
    ranked = equivalent_order(best, "rank_ic", "n_parameters")
    finalists = ranked["config"].head(2).tolist()
    return table, ranked, finalists + [name for name in ["control"] if name not in finalists]


if RUN_FINALIST_SELECTION:
    scores = screening_scores()
    table, ranked, finalists = choose_finalists(scores)
    save_table(scores, "screening_runs")
    save_table(table.sort_values("rank_ic", ascending=False), "screening_leaderboard")
    display(ranked)
    chosen = {"finalists": finalists, "scores": short_hash(json.loads(table.drop(columns="seconds").to_json())), "code": CODE_VERSION}
    if FINALISTS_FILE.exists() and json.loads(FINALISTS_FILE.read_text()) != chosen:
        raise ValueError("finalists.json was already written from other results.")
    save_json(FINALISTS_FILE, chosen)
    print(f"Finalists: {finalists}")

## 13. Finalist Training across Complete Intended Datasets

Each finalist is trained in the `final` context: U.S. months from February 2000 to December 2017, both countries, all windows and three seeds. No
subsets are used.

In [ ]:
def load_finalists():
    if not FINALISTS_FILE.exists():
        raise FileNotFoundError("finalists.json is missing: run screening and finalist selection first.")
    return json.loads(FINALISTS_FILE.read_text())["finalists"]


if RUN_FINAL_TRAINING:
    for position, config in enumerate(load_finalists()):
        if ONLY_FINALIST is None or position == ONLY_FINALIST:
            run_configuration("final", config, FINAL_SEEDS)

## 14. Validation-Based Model Selection

**Entries.** Every finalist × strategy, the three baselines, and one blend per finalist. A neural entry's forecast is the average of its three seeds. The blend is
the equal-weight average of within-month percentile ranks of `finetuned`, `joint` and `xgboost_rank`. It is fixed in advance and has no fitted weights.

**Primary metric.** Mean monthly validation rank IC, equal weight over the six country–window cells, full universe. The column `mean_seed_ic`
averages the individual seeds. The Version 2 references from the audit (`rank`, fine-tuned) are 0.1328 for the seed average and 0.1274 for the
average of seeds; their 24 months are the same and were already examined. Baselines have no counted parameters, so they win ties.

**Rule, applied before any test forecast exists.** Among entries whose calibrated raw and clipped MSE are each within 0.25% of a zero forecast in both countries, take the
highest IC; entries within 0.001 are tied and the one with fewer parameters, then the smaller seed variation, is chosen. Entries that rank well
but have materially worse raw or calibrated MSE are flagged. Pairwise tests compare each entry with the chosen one by a circular block bootstrap
(6-month blocks) on the 24 monthly averages of the six cells, with Holm adjustment over all entries. With 24 reused months these tests
have little power; a large p-value means the data cannot separate the models, not that they are equal.

In [ ]:
BLEND = ["finetuned", "joint", "xgboost_rank"]
RANK_UNITS = ["blend", "xgboost_rank"]  # forecasts in rank units: uncalibrated MSE is not meaningful
SELECTION_FILE = RESULTS_DIR / "selection.json"


def entries(finalists):
    """(configuration, strategy) of every candidate forecast."""
    return [("baselines", family) for family in BASELINES] + list(product(finalists, STRATEGIES + ["blend"]))


def seed_list(config):
    return [SEED] if config == "baselines" else FINAL_SEEDS


def seed_forecast(config, strategy, country, window, seed, period):
    key = ("final", config, strategy, country, window, seed)
    return load_forecast(key, "score") if period == "validation" else load_test_forecast(key)


def entry_forecast(config, strategy, country, window, period):
    """Deployable forecast: seed average, or the blend of three averages."""
    if strategy == "blend":
        parts = [entry_forecast("baselines" if name in BASELINES else config, name, country, window, period) for name in BLEND]
        parts = [part.assign(prediction=part.groupby("eom")["prediction"].rank(pct=True) - 0.5) for part in parts]
    else:
        parts = [seed_forecast(config, strategy, country, window, seed, period) for seed in seed_list(config)]
    merged = parts[0]
    for i, part in enumerate(parts[1:]):
        merged = merged.merge(part, on=KEYS, validate="one_to_one", how="outer", suffixes=("", f"_{i}"), indicator=True)
        if (merged.pop("_merge") != "both").any():
            raise ValueError(f"{config} {strategy} {country} {window}: forecasts cover different stocks.")
    values = merged[[name for name in merged.columns if name.startswith("prediction")]]
    return merged[KEYS].assign(prediction=values.mean(axis=1))


def validation_monthly(finalists):
    """Monthly validation scores of every entry (all metrics) and of every single seed (rank IC)."""
    entry_rows, seed_rows = [], []
    for (config, strategy), country, window in product(entries(finalists), LOCAL_COUNTRIES, WINDOWS):
        label = {"config": config, "strategy": strategy, "country": country, "window": window}
        limits = local_limits("final", country, window)
        table = add_actuals(entry_forecast(config, strategy, country, window, "validation"), country)
        entry_rows.append(monthly_scores(calibrate_validation(table, limits), limits).assign(**label))
        for seed in seed_list(config) if strategy != "blend" else []:
            table = add_actuals(seed_forecast(config, strategy, country, window, seed, "validation"), country)
            record = load_record(("final", config, strategy, country, window, seed))
            scores = monthly_scores(calibrate_validation(table, limits), limits)
            for universe, part in scores.groupby("universe"):
                seed_rows.append({**label, "seed": seed, "universe": universe, **summarize_scores(part),
                                  "n_parameters": record["n_parameters"], "seconds": record.get("total_seconds", record["seconds"])})
    return pd.concat(entry_rows, ignore_index=True), pd.DataFrame(seed_rows)


def cell_summary(monthly):
    names = ["config", "strategy", "country", "window", "universe"]
    return pd.DataFrame([{**dict(zip(names, group)), **summarize_scores(part)} for group, part in monthly.groupby(names)])


def build_leaderboard(monthly, seeds):
    """One row per entry: the single table used for selection."""
    cells, names = cell_summary(monthly), ["config", "strategy"]
    full = cells[cells["universe"] == "full"]
    table = full.groupby(names).agg(
        rank_ic=("rank_ic", lambda v: v.mean(skipna=False)), worst_cell_ic=("rank_ic", "min"), pearson_ic=("pearson_ic", "mean"),
        mse_gain_raw=("mse_gain_raw", "mean"), mse_gain_calibrated=("mse_gain_calibrated", "mean"), clipped_gain_calibrated=("clipped_gain_calibrated", "mean"),
    )
    for country in LOCAL_COUNTRIES:
        part = full[full["country"] == country].groupby(names)
        table[f"ic_{country}"], table[f"mse_gain_{country}"] = part["rank_ic"].mean(), part["mse_gain_calibrated"].mean()
        table[f"clipped_gain_{country}"] = part["clipped_gain_calibrated"].mean()
        table[f"raw_gain_{country}"] = part["mse_gain_raw"].mean()
    rank_units = table.index.get_level_values("strategy").isin(RANK_UNITS)
    table.loc[rank_units, [name for name in table.columns if "raw" in name]] = np.nan
    table["ic_investable"] = cells[cells["universe"] == "investable"].groupby(names)["rank_ic"].mean()
    by_seed = seeds[seeds["universe"] == "full"]
    table["mean_seed_ic"] = by_seed.groupby(names)["rank_ic"].mean()
    table["seed_std"] = by_seed.groupby(names + ["seed"])["rank_ic"].mean().groupby(names).std()
    table["n_parameters"] = by_seed.groupby(names)["n_parameters"].max()
    table["training_minutes"] = by_seed.groupby(names)["seconds"].sum() / 60
    for config, strategy in table.index:
        if strategy == "blend":  # its members: two sets of networks and the XGBoost model
            members = [("baselines" if name in BASELINES else config, name) for name in BLEND]
            table.loc[(config, strategy), "n_parameters"] = sum(table.loc[member, "n_parameters"] for member in members)
            table.loc[(config, strategy), "training_minutes"] = sum(table.loc[member, "training_minutes"] for member in members)
    worst = table[[f"{kind}_gain_{country}" for kind, country in product(["mse", "clipped"], LOCAL_COUNTRIES)]].min(axis=1)
    worst_raw = table[[f"raw_gain_{country}" for country in LOCAL_COUNTRIES]].min(axis=1)
    table["eligible"] = (worst >= -MSE_TOLERANCE) & table["rank_ic"].notna()
    table["flag"] = np.select([worst < -MSE_TOLERANCE, worst_raw < -MSE_TOLERANCE],
                              ["calibrated MSE materially worse than zero", "uncalibrated MSE materially worse than zero"], "")
    return table.reset_index().sort_values("rank_ic", ascending=False, ignore_index=True)


def choose_entry(table):
    eligible = table[table["eligible"]].assign(seed_std=lambda t: t["seed_std"].fillna(np.inf))
    if eligible.empty:
        raise ValueError("No entry satisfies the MSE rule; do not generate test forecasts.")
    return equivalent_order(eligible, "rank_ic", "n_parameters", "seed_std").iloc[0]

In [ ]:
def block_indices(n_months, rng):
    """Circular block-bootstrap month positions."""
    starts = rng.integers(0, n_months, size=(BOOTSTRAP_SAMPLES, int(np.ceil(n_months / BLOCK_MONTHS))))
    return ((starts[:, :, None] + np.arange(BLOCK_MONTHS)) % n_months).reshape(BOOTSTRAP_SAMPLES, -1)[:, :n_months]


def paired_test(a, b, seed=SEED):
    """Mean difference of two monthly series with a 95% interval and a two-sided p-value."""
    difference = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if not np.isfinite(difference).all():
        return {"difference": np.nan, "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    samples = difference[block_indices(len(difference), np.random.default_rng(seed))].mean(axis=1)
    centered = samples - difference.mean()  # distribution under no difference
    return {"difference": difference.mean(), "ci_low": np.quantile(samples, 0.025), "ci_high": np.quantile(samples, 0.975),
            "p_value": (1 + (np.abs(centered) >= abs(difference.mean())).sum()) / (1 + len(samples))}


def holm_adjust(p_values):
    valid = p_values.dropna().sort_values()
    adjusted = (valid * np.arange(len(valid), 0, -1)).cummax().clip(upper=1)
    return adjusted.reindex(p_values.index)


def entry_series(monthly, column="rank_ic"):
    """Monthly average over the six cells of every entry (months by entries)."""
    full = monthly[monthly["universe"] == "full"]
    cells = full.set_index(["predicted_month", "config", "strategy", "country", "window"])[column].unstack(["config", "strategy", "country", "window"])
    return pd.DataFrame({name: cells[name].mean(axis=1, skipna=False) for name in dict.fromkeys((c, s) for c, s, _, _ in cells.columns)})


def selection_tests(monthly, chosen):
    series = entry_series(monthly)
    rows = [{"config": c, "strategy": s, **paired_test(series[chosen], series[c, s])} for c, s in series.columns if (c, s) != chosen]
    table = pd.DataFrame(rows)
    table["p_holm"] = holm_adjust(table["p_value"])
    return table.rename(columns={"difference": "chosen_minus_entry"})


def lock_selection():
    """Build the leaderboard and record the choice; an existing different choice raises an error."""
    finalists = load_finalists()
    monthly, seeds = validation_monthly(finalists)
    table = build_leaderboard(monthly, seeds)
    chosen = choose_entry(table)
    save_table(table, "validation_leaderboard")
    save_table(cell_summary(monthly), "validation_by_cell")
    save_table(seeds, "validation_by_seed")
    save_table(selection_tests(monthly, (chosen["config"], chosen["strategy"])), "validation_tests")
    selection = {"config": chosen["config"], "strategy": chosen["strategy"], "finalists": finalists, "code": CODE_VERSION,
                 "leaderboard": short_hash(json.loads(table.drop(columns="training_minutes").round(10).to_json()))}
    if SELECTION_FILE.exists() and json.loads(SELECTION_FILE.read_text()) != selection:
        raise ValueError("selection.json was already locked from other results.")
    save_json(SELECTION_FILE, selection)
    display(table)
    print(f"Selected on validation only: {selection['config']} | {selection['strategy']}")


if RUN_PREDICTIONS:
    lock_selection()

## 15. Prediction Generation

Test forecasts are generated only after `selection.json` exists. Models are loaded through their verified records; forecasts cover all screened
stocks, whether or not a return was realized. These files contain licensed stock-level data and must stay private.

In [ ]:
def load_selection():
    if not SELECTION_FILE.exists():
        raise FileNotFoundError("selection.json is missing: the model must be selected on validation data before any test forecast.")
    return json.loads(SELECTION_FILE.read_text())


def test_path(key, suffix):
    return RESULTS_DIR / "test_forecasts" / run_path(*key, suffix).relative_to(RUNS_DIR)


def source_run(key):
    """Run that holds the model of a forecast, and its model file type."""
    if key[1] == "baselines":
        return key, ".joblib"
    return (source_key(key) if key[2] == "direct" else key), ".pt"


def model_sha(key):
    source, suffix = source_run(key)
    return load_record(source)["files"][run_path(*source, suffix).name]


def load_test_forecast(key):
    record, path = json.loads(test_path(key, ".json").read_text()), test_path(key, ".parquet")
    if record["sha256"] != file_sha256(path) or record["model_sha256"] != model_sha(key):
        raise ValueError(f"{path}: damaged, or produced by another model.")
    return pd.read_parquet(path)


def predict_test(key, months):
    if test_path(key, ".json").exists():
        load_test_forecast(key)  # verified; an inconsistent file raises an error
        return
    source, suffix = source_run(key)
    expected = model_sha(key)
    if suffix == ".joblib":
        model = joblib.load(run_path(*source, suffix))
        predictions = [np.asarray(model.predict(month["X"].numpy())) for month in months]
    else:
        saved = load_checkpoint(source)
        model = make_model(saved["spec"], saved["columns"])
        model.load_state_dict(saved["state"])
        predictions = predict_months(model, months)
    table = forecast_table(months, predictions, "test").drop(columns="block")
    if not np.isfinite(table["prediction"]).all():
        raise ValueError(f"{key}: nonfinite test forecasts.")
    save_safely(test_path(key, ".parquet"), lambda temporary: table.to_parquet(temporary, index=False))
    save_json(test_path(key, ".json"), {"sha256": file_sha256(test_path(key, ".parquet")), "model_sha256": expected, "environment": environment_details()})


if RUN_PREDICTIONS:
    finalists = load_selection()["finalists"]
    for country in LOCAL_COUNTRIES:
        months = load_months(country, *TEST, allow_test=True)
        for config, strategy in entries(finalists):
            for window, seed in product(WINDOWS, seed_list(config)) if strategy != "blend" else []:
                predict_test(("final", config, strategy, country, window, seed), months)
        print(f"{country}: test forecasts complete")

## 16. Statistical Evaluation

Test months are January 2020 – December 2025 and are **exploratory**. Calibration slopes come from 2018–2019. The entries and comparisons below were
fixed before Version 3 test results. Each comparison is tested per country and window by a circular block bootstrap (72 months), with Holm
adjustment within one family of (number of comparisons × 6) rank-IC tests. Calibrated MSE against a zero forecast is a second, separate family.

In [ ]:
def test_monthly(finalists):
    """Monthly test scores of every entry."""
    rows = []
    for (config, strategy), country, window in product(entries(finalists), LOCAL_COUNTRIES, WINDOWS):
        limits = local_limits("final", country, window)
        fit = add_actuals(entry_forecast(config, strategy, country, window, "validation"), country)
        table = add_actuals(entry_forecast(config, strategy, country, window, "test"), country)
        scores = monthly_scores(calibrate(fit, table, limits), limits)
        if scores.loc[scores["universe"] == "full", "predicted_month"].nunique() != 72:
            raise ValueError(f"{config} {strategy} {country} {window}: the test period must have 72 months.")
        rows.append(scores.assign(config=config, strategy=strategy, country=country, window=window))
    return pd.concat(rows, ignore_index=True)


def test_comparisons(selection):
    """Predeclared references for the selected entry, and the transfer question within its configuration."""
    chosen = (selection["config"], selection["strategy"])
    network = chosen[0] if chosen[0] != "baselines" else "control"
    pairs = [(chosen, reference) for reference in [
        ("control", "finetuned"), (network, "local"), ("baselines", "xgboost"), ("baselines", "xgboost_rank")]]
    pairs += [((network, strategy), (network, "local")) for strategy in ["direct", *TRANSFER]]
    return list(dict.fromkeys(pair for pair in pairs if pair[0] != pair[1]))


def test_statistics(monthly, selection):
    full = monthly[monthly["universe"] == "full"]
    series = full.set_index(["predicted_month", "config", "strategy", "country", "window"])[["rank_ic", "mse_calibrated", "mse_zero"]]
    series = series.unstack(["config", "strategy", "country", "window"])
    rows = []
    for (a, b), country, window in product(test_comparisons(selection), LOCAL_COUNTRIES, WINDOWS):
        rows.append({"family": "rank IC", "a": " ".join(a), "b": " ".join(b), "country": country, "window": window,
                     **paired_test(series["rank_ic"][(*a, country, window)], series["rank_ic"][(*b, country, window)])})
    chosen = (selection["config"], selection["strategy"])
    for country, window in product(LOCAL_COUNTRIES, WINDOWS):  # positive difference = lower error than the zero forecast
        rows.append({"family": "calibrated MSE", "a": " ".join(chosen), "b": "zero forecast", "country": country, "window": window,
                     **paired_test(series["mse_zero"][(*chosen, country, window)], series["mse_calibrated"][(*chosen, country, window)])})
    table = pd.DataFrame(rows)
    table["p_holm"] = table.groupby("family")["p_value"].transform(holm_adjust)
    return table


if RUN_FINAL_EVALUATION:
    selection = load_selection()
    test_scores = test_monthly(selection["finalists"])
    test_cells, test_table = cell_summary(test_scores), test_statistics(test_scores, selection)
    save_table(test_scores, "test_monthly")
    save_table(test_cells, "test_by_cell")
    save_table(test_table, "test_comparisons")
    display(test_cells[(test_cells["config"] == selection["config"]) & (test_cells["strategy"] == selection["strategy"])])
    display(test_table)

## 17. Portfolio Construction and Robustness

Secondary to the forecasting question. At each formation date, stocks are sorted by the forecast: $1 long in the highest quantile and $1 short in
the lowest, equal weights, 10% (primary) or 20%, in the full or investable universe.

- **Weights are intended weights** and are never rescaled after seeing which returns were realized.
- **Missing returns:** `exact` is defined only if every selected stock has a return. Three labeled assumptions fill the unobserved positions:
  `zero`; `minus30` (−30% for all); `adverse` (−30% for longs, +30% for shorts). Coverage is reported.
- **Turnover** is the traded notional from the previous weights after they drift with returns. Costs are 10 or 25 bps per dollar traded; stocks that
  disappear are traded out. Costs of entering the first month are included; the final unwinding is not.
- **Outliers:** Sharpe ratios are also shown without the largest absolute month. Realized returns are never winsorized.
- **Drawdown** compounds the monthly profit on $1 of capital per side; it is undefined once the capital is lost.

In [ ]:
QUANTILES, COSTS = [0.10, 0.20], [10, 25]
ASSUMPTIONS = {"zero": (0.0, 0.0), "minus30": (-0.30, -0.30), "adverse": (-0.30, 0.30)}  # (long, short)


def long_short_weights(month, quantile):
    """Intended weights by stock; no position if the forecasts do not vary."""
    if month["prediction"].nunique() < 2:
        return pd.Series(dtype="float64")
    ids = month.sort_values(["prediction", "id"])["id"].to_numpy()
    n = max(1, int(np.floor(quantile * len(ids))))
    return pd.Series([-1 / n] * n + [1 / n] * n, index=np.concatenate([ids[:n], ids[-n:]]))


def portfolio_months(table, quantile, universe):
    """Monthly returns under every assumption, with turnover and missing-return coverage."""
    if universe == "investable":
        table = table[table["investable"]]
    previous, rows = {name: pd.Series(dtype="float64") for name in ASSUMPTIONS}, []
    for eom, month in table.groupby("eom"):
        weights = long_short_weights(month, quantile)
        returns = month.set_index("id")[TARGET].reindex(weights.index)
        observed = returns.notna()
        row = {"predicted_month": eom + pd.offsets.MonthEnd(1), "stocks_per_side": int((weights > 0).sum()),
               "unobserved_exposure": weights[~observed].abs().sum() / 2, "fully_observed": bool(observed.all()),
               "largest_contribution": (weights * returns).abs().max()}
        for name, (long_value, short_value) in ASSUMPTIONS.items():
            filled = returns.where(observed, pd.Series(np.where(weights > 0, long_value, short_value), index=weights.index))
            row[f"gross_{name}"] = (weights * filled).sum()
            row[f"turnover_{name}"] = weights.sub(previous[name], fill_value=0).abs().sum()
            previous[name] = weights * (1 + filled)
        row["gross_exact"] = row["gross_zero"] if row["fully_observed"] else np.nan
        row["turnover_exact"] = row["turnover_zero"]  # drift of a fully measured month is identical; gaps drift with zero
        rows.append(row)
    return pd.DataFrame(rows)


def sharpe_ratio(returns):
    spread = np.std(returns, ddof=1) if len(returns) > 1 else 0.0
    return float(np.mean(returns) / spread * np.sqrt(12)) if spread > 0 else np.nan


def compounded_drawdown(returns):
    """Worst loss from a previous peak of $1 compounding; undefined if the capital is lost."""
    wealth = np.concatenate([[1.0], np.cumprod(1 + returns)])
    if (wealth <= 0).any():
        return np.nan
    return float((wealth / np.maximum.accumulate(wealth) - 1).min())


def portfolio_summary(months):
    rows = []
    for name, cost in product(["exact", *ASSUMPTIONS], COSTS):
        net = (months[f"gross_{name}"] - months[f"turnover_{name}"] * cost / 10_000).to_numpy()
        valid = net[np.isfinite(net)]
        keep = np.arange(len(valid)) != np.argmax(np.abs(valid)) if len(valid) else []
        rows.append({
            "accounting": name, "cost_bps": cost, "months": len(net), "months_measured": len(valid),
            "net_mean_monthly": valid.mean() if len(valid) else np.nan, "net_sharpe": sharpe_ratio(valid),
            "net_sharpe_without_largest_month": sharpe_ratio(valid[keep]) if len(valid) else np.nan, "worst_month": valid.min() if len(valid) else np.nan,
            "drawdown": compounded_drawdown(valid) if len(valid) == len(net) else np.nan, "mean_turnover": months[f"turnover_{name}"].mean(),
            "mean_unobserved_exposure": months["unobserved_exposure"].mean(), "months_fully_observed": int(months["fully_observed"].sum()),
            "largest_single_stock_contribution": months["largest_contribution"].max(), "mean_stocks_per_side": months["stocks_per_side"].mean(),
            "note": "" if len(valid) == len(net) else "only fully observed months; not a 72-month result",
        })
    return pd.DataFrame(rows)


def portfolio_entries(selection):
    chosen = (selection["config"], selection["strategy"])
    network = chosen[0] if chosen[0] != "baselines" else "control"
    return list(dict.fromkeys([chosen, ("control", "finetuned"), (network, "local"), ("baselines", "xgboost"), ("baselines", "xgboost_rank")]))


if RUN_FINAL_EVALUATION:
    summaries, series = [], []
    for (config, strategy), country, window in product(portfolio_entries(selection), LOCAL_COUNTRIES, WINDOWS):
        table = add_actuals(entry_forecast(config, strategy, country, window, "test"), country)
        for universe, quantile in product(["full", "investable"], QUANTILES):
            label = {"config": config, "strategy": strategy, "country": country, "window": window, "universe": universe, "quantile": quantile}
            months = portfolio_months(table, quantile, universe)
            series.append(months.assign(**label))
            summaries.append(portfolio_summary(months).assign(**label))
    portfolios = pd.concat(summaries, ignore_index=True)
    save_table(portfolios, "portfolio_summary")
    save_table(pd.concat(series, ignore_index=True), "portfolio_monthly")
    primary = portfolios[(portfolios["quantile"] == 0.10) & (portfolios["cost_bps"] == 10) & (portfolios["accounting"] == "zero")]
    display(primary.pivot_table(index=["config", "strategy", "country", "window"], columns="universe", values=["net_sharpe", "net_sharpe_without_largest_month"]))

## 18. Final Comparisons, Visualizations and Conclusions

The figure shows rank IC by country and window for the selected entry and its references. Conclusions are written after training; until then
**no Version 3 improvement is claimed**. When the results exist, read them in this order:

1. Validation leaderboard: is the selected entry ahead of `control` by more than its seed variation, and is its MSE flag empty?
2. Test comparisons: which Holm-adjusted differences survive, and in which countries and windows?
3. Investable universe: does the ranking survive without micro and nano stocks?
4. Portfolios: do results depend on one month, on missing returns or on the universe?

The test period has been examined repeatedly, so a confirmed improvement needs returns after December 2025.

In [ ]:
def plot_rank_ic(cells, selection):
    shown = portfolio_entries(selection)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        for config, strategy in shown:
            part = cells[(cells["config"] == config) & (cells["strategy"] == strategy) & (cells["country"] == country) & (cells["universe"] == "full")]
            values = part.set_index("window")["rank_ic"].reindex(list(WINDOWS))
            axis.plot(range(len(WINDOWS)), values, marker="o", label=f"{config} {strategy}")
        axis.set_xticks(range(len(WINDOWS)), ["1 year", "3 years", "5 years"])
        axis.set_title({"GBR": "United Kingdom", "JPN": "Japan"}[country])
        axis.set_xlabel("Local training history")
    axes[0].set_ylabel("Mean monthly rank IC, 2020–2025")
    axes[1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_safely(TABLES_DIR / "test_rank_ic.png", lambda temporary: fig.savefig(temporary, dpi=150, format="png"))
    plt.show()


if RUN_FINAL_EVALUATION:
    plot_rank_ic(test_cells, selection)
    leaderboard = pd.read_csv(TABLES_DIR / "validation_leaderboard.csv")
    final = leaderboard[["config", "strategy", "rank_ic", "mean_seed_ic", "seed_std", "flag"]].merge(
        test_cells[test_cells["universe"] == "full"].groupby(["config", "strategy"])[["rank_ic", "mse_gain_calibrated"]].mean().reset_index(),
        on=["config", "strategy"], suffixes=("_validation", "_test"))
    save_table(final, "final_comparison")
    display(final)
print("Notebook finished.")